In [ ]:
# Bitcoin AI Predictor – Data Analytics Analysis Notebook
# Author: Aryan Kaushik
# Purpose: End-to-end data analytics workflow for Bitcoin price direction prediction

# ============================
# 1. Imports & Configuration
# ============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

plt.rcParams['figure.figsize'] = (10, 5)

# ============================
# 2. Data Loading
# ============================
# NOTE: Replace path with actual dataset file

df = pd.read_csv('data/bitcoin_data.csv')
df.head()

# ============================
# 3. Data Cleaning
# ============================

df.columns = df.columns.str.lower()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Check missing values
df.isna().sum()

# ============================
# 4. Feature Engineering
# ============================

# Daily returns
df['return'] = df['close'].pct_change()

# Rolling features
df['ma_7'] = df['close'].rolling(7).mean()
df['ma_21'] = df['close'].rolling(21).mean()
df['volatility_7'] = df['return'].rolling(7).std()

# Target variable: 1 if next-day return positive, else 0
df['target'] = (df['return'].shift(-1) > 0).astype(int)

# Drop NaNs created by rolling windows
df = df.dropna().reset_index(drop=True)

# ============================
# 5. Exploratory Data Analysis (EDA)
# ============================

# Price trend
plt.plot(df['date'], df['close'])
plt.title('Bitcoin Price Over Time')
plt.xlabel('Date')
plt.ylabel('Price')
plt.show()

# Returns distribution
plt.hist(df['return'], bins=50)
plt.title('Distribution of Daily Returns')
plt.show()

# Correlation heatmap
sns.heatmap(df[['return', 'ma_7', 'ma_21', 'volatility_7']].corr(), annot=True)
plt.title('Feature Correlation Matrix')
plt.show()

# ============================
# 6. Modeling Preparation
# ============================

features = ['ma_7', 'ma_21', 'volatility_7']
X = df[features]
y = df['target']

# Time-based split
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================
# 7. Baseline Model
# ============================

baseline_accuracy = max(y_test.mean(), 1 - y_test.mean())
baseline_accuracy

# ============================
# 8. Logistic Regression Model
# ============================

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred))

# ============================
# 9. Model Evaluation
# ============================

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.show()

# ============================
# 10. Cross-Validation (Time Series)
# ============================

tscv = TimeSeriesSplit(n_splits=5)
cv_scores = cross_val_score(model, scaler.fit_transform(X), y, cv=tscv)
cv_scores

# ============================
# 11. Key Takeaways
# ============================
# - Model performance is only slightly above baseline
# - Signals are weak and unstable across time
# - Confirms market efficiency in short-term crypto prices

# ============================
# 12. Conclusion
# ============================
# This notebook demonstrates a realistic data analytics workflow
# with honest evaluation rather than over-optimistic claims.
